# LangGraph with AgentCore Memory Tool (Short term memory)

# LangGraph 与 AgentCore Memory 工具（短期记忆）

## Introduction

## 简介

This notebook demonstrates how to integrate Amazon Bedrock AgentCore Memory capabilities with a conversational AI agent using LangGraph framework. We'll focus on **short-term memory** retention within a single conversation session - allowing an agent to recall information from earlier in the conversation without explicit context management.

本笔记本演示如何使用 LangGraph 框架将 Amazon Bedrock AgentCore Memory 功能与对话式 AI 代理集成。我们将重点关注单个对话会话中的**短期记忆**保留 - 允许代理在无需显式上下文管理的情况下回忆对话中较早的信息。


## Tutorial Details

## 教程详情

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Short Term Conversational                                                        |
| Agent usecase       | Personal Fitness                                                                 |
| Agentic Framework   | Langgraph                                                                        |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| Tutorial components | AgentCore Short-term Memory, Langgraph, Memory retrieval via Tool                |
| Example complexity  | Beginner                                                                         |

| 信息                 | 详情                                                                              |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型             | 短期对话                                                                          |
| 代理用例             | 个人健身                                                                          |
| 代理框架             | Langgraph                                                                        |
| LLM 模型            | Anthropic Claude Haiku 4.5                                                      |
| 教程组件             | AgentCore 短期记忆、Langgraph、通过工具进行记忆检索                                  |
| 示例复杂度           | 初级                                                                              |

You'll learn to:
- Create a memory store with AgentCore Memory for short-term memory
- Use LangGraph to create an agent with structured memory workflows
- Implement memory tools for conversation history retrieval
- Access and utilize contextual information within a single session
- Enhance conversational experiences through effective memory recall

您将学习：
- 使用 AgentCore Memory 创建用于短期记忆的内存存储
- 使用 LangGraph 创建具有结构化记忆工作流的代理
- 实现用于对话历史检索的记忆工具
- 在单个会话中访问和利用上下文信息
- 通过有效的记忆回忆增强对话体验


### Scenario Context

### 场景背景

In this example, we'll create a "**Personal Fitness Coach**" that can remember workout details, fitness goals, physical limitations, and exercise preferences as they are mentioned throughout the conversation. This assistant will demonstrate how effective short-term memory management enables a more natural and personalized fitness coaching experience without requiring users to repeatedly state their information.

在本示例中，我们将创建一个"**个人健身教练**"，它可以记住在整个对话过程中提到的锻炼细节、健身目标、身体限制和运动偏好。这个助手将演示有效的短期记忆管理如何实现更自然和个性化的健身指导体验，而无需用户反复陈述他们的信息。


## Architecture

## 架构

<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## Prerequisites

## 前提条件

- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

- Python 3.10+
- 具有适当权限的 AWS 账户
- 具有 AgentCore Memory 适当权限的 AWS IAM 角色
- 访问 Amazon Bedrock 模型

Let's get started by setting up our environment!

让我们开始设置我们的环境！

## Step 1: Environment Setup

## 步骤 1：环境设置

Let's begin importing all the necessary libraries and defining the clients to make this notebook work.

让我们开始导入所有必要的库并定义客户端，以使本笔记本正常工作。

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime

Define the region and the role with the appropiate permissions for Amazon Bedrock models and AgentCore

定义具有 Amazon Bedrock 模型和 AgentCore 适当权限的区域和角色

In [ ]:
import os
region = os.getenv('AWS_REGION', 'us-west-2')

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

### How the Integration Works

### 集成工作原理

The integration between LangGraph and AgentCore Memory involves:

1. Using AgentCore Memory to store conversations in the short term memory
2. Structured workflows in LangGraph to manage memory operations

LangGraph 与 AgentCore Memory 之间的集成包括：

1. 使用 AgentCore Memory 在短期记忆中存储对话
2. 在 LangGraph 中使用结构化工作流来管理记忆操作

This approach separates memory management from reasoning, creating a cleaner and more maintainable agent architecture.

这种方法将记忆管理与推理分离，创建更清晰、更易维护的代理架构。

## Step 2: Memory Creation

## 步骤 2：创建记忆

In this section, we'll create a memory store using the AgentCore Memory SDK. This memory store will allow our agent to retain information from the conversation.

在本节中，我们将使用 AgentCore Memory SDK 创建一个内存存储。这个内存存储将允许我们的代理保留对话中的信息。

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from botocore.exceptions import ClientError

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "FitnessCoach"
memory_id = None

In [ ]:
try:
    print("Creating Memory...")
    # Create the memory resource
    memory = client.create_memory_and_wait(
        name=memory_name,                       # This name is unique across all memories in this account
        description="Fitness Coach Agent",      # Human-readable description
        strategies=[],                          # No memory strategies for short-term memory
        event_expiry_days=7,                    # Memories expire after 7 days
        max_wait=300,                           # Maximum time to wait for memory creation (5 minutes)
        poll_interval=10                        # Check status every 10 seconds
    )

    # Extract and print the memory ID
    memory_id = memory['id']
    logger.info(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.info(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

## Step 3: LangGraph Agent Creation

## 步骤 3：创建 LangGraph 代理

Let's import all the libraries we need to create the agent with LangGraph.

让我们导入使用 LangGraph 创建代理所需的所有库。

In [ ]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_aws import ChatBedrock

### LangGraph Agent Implementation

### LangGraph 代理实现

Now let's create the agent with LangGraph, incorporating our memory tools:

现在让我们使用 LangGraph 创建代理，并整合我们的记忆工具：

In [ ]:
def create_agent(client, memory_id, actor_id, session_id):
    """Create and configure the LangGraph agent"""
    
    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )
    
    @tool
    def list_events():
        """Tool used when needed to retrieve recent information""" 
        events = client.list_events(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                max_results=10
            )
        return events
        
    
    # Bind tools to the LLM
    tools = [list_events]
    llm_with_tools = llm.bind_tools(tools)
    
    # System message
    system_message = """You are the Personal Fitness Coach, a sophisticated fitness guidance assistant.
                        PURPOSE:
                        - Help users develop workout routines based on their fitness goals
                        - Remember user's exercise preferences, limitations, and progress
                        - Provide personalized fitness recommendations and training plans
                        MEMORY CAPABILITIES:
                        - You have access to recent events with the list_events tool
                        """
    
    # Define the chatbot node
    def chatbot(state: MessagesState):
        raw_messages = state["messages"]
    
        # Remove any existing system messages to avoid duplicates or misplacement
        non_system_messages = [msg for msg in raw_messages if not isinstance(msg, SystemMessage)]
    
        # Always ensure SystemMessage is first
        messages = [SystemMessage(content=system_message)] + non_system_messages
    
        latest_user_message = next((msg.content for msg in reversed(messages) if isinstance(msg, HumanMessage)), None)
    
        # Get response from model with tools bound
        response = llm_with_tools.invoke(messages)
    
        # Save conversation if applicable
        if latest_user_message and response.content.strip():  # Check that response has content
            conversation = [
                (latest_user_message, "USER"),
                (response.content, "ASSISTANT")
            ]
            
            # Validate that all message texts are non-empty
            if all(msg[0].strip() for msg in conversation):  # Ensure no empty messages
                try:
                    client.create_event(
                        memory_id=memory_id,
                        actor_id=actor_id,
                        session_id=session_id,
                        messages=conversation
                    )
                except Exception as e:
                    print(f"Error saving conversation: {str(e)}")
        
        # Append response to full message history
        return {"messages": raw_messages + [response]}
    
    # Create the graph
    graph_builder = StateGraph(MessagesState)
    
    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # Set entry point
    graph_builder.set_entry_point("chatbot")
    
    # Compile the graph
    return graph_builder.compile()

### Creating a Wrapper for Agent Invocation

### 创建代理调用的包装器

Let's create a simple wrapper to invoke our agent:

让我们创建一个简单的包装器来调用我们的代理：

In [ ]:
def langgraph_bedrock(payload, agent):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    
    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # Extract the final message content
    return response["messages"][-1].content

## Step 4: Run the LangGraph Agent

## 步骤 4：运行 LangGraph 代理

We can now run the agent with our AgentCore Memory integration.

现在我们可以使用 AgentCore Memory 集成来运行代理。

In [ ]:
# Create unique actor and session IDs for this conversation
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"workout-{datetime.now().strftime('%Y%m%d%H%M%S')}"

In [ ]:
# Create the agent with AgentCore Memory integration
agent = create_agent(client, memory_id, actor_id, session_id)

#### Congratulations ! Your Agent is ready !!

#### 恭喜！您的代理已准备就绪！！

### Let's test the Agent

### 让我们测试代理

Let's interact with our agent to test its memory capabilities:

让我们与代理交互以测试其记忆能力：

In [ ]:
response = langgraph_bedrock({"prompt": "Hello! This is my first day, I need a workout routine."}, agent)
print(f"Agent: {response}\n")

In [ ]:
response = langgraph_bedrock({"prompt": "I want to build muscle, looking for a biceps routine. I have some lower back problems."}, agent)
print(f"Agent: {response}\n")

In [ ]:
response = langgraph_bedrock({"prompt": "Can you give me three exercises with number of reps?"}, agent)
print(f"Agent: {response}\n")

### Testing Memory Persistence

### 测试记忆持久化

To truly demonstrate the power of the AgentCore Memory integration, let's create a new agent instance and see if it can recall our previous conversation:

为了真正展示 AgentCore Memory 集成的强大功能，让我们创建一个新的代理实例，看看它是否可以回忆起我们之前的对话：

In [ ]:
# Create a new agent instance (simulating a new session)
new_agent = create_agent(client, memory_id, actor_id, session_id)

# Test if the new agent remembers our preferences
response = langgraph_bedrock({
    "prompt": "Hello again! Can you remind me about my last workout session?"
}, new_agent)

print("New Agent Session:\n")
print(f"Agent: {response}")

## Summary

## 总结

In this notebook, we've demonstrated:

在本笔记本中，我们演示了：

1. How to create a AgentCore Memory resource for an AI agent
2. Building a LangGraph workflow with memory integration
3. Implementing memory tools for conversation history retrieval
4. Creating an agent that intelligently uses memory when needed
5. Testing memory persistence across agent instances

1. 如何为 AI 代理创建 AgentCore Memory 资源
2. 使用记忆集成构建 LangGraph 工作流
3. 实现用于对话历史检索的记忆工具
4. 创建在需要时智能使用记忆的代理
5. 测试跨代理实例的记忆持久化

This integration showcases the power of combining structured workflows (LangGraph) with robust memory systems (AgentCore Memory) to create more intelligent and context-aware AI agents.

这种集成展示了将结构化工作流（LangGraph）与强大的记忆系统（AgentCore Memory）相结合的能力，以创建更智能、更具上下文感知能力的 AI 代理。

The approach we've demonstrated can be extended to more complex use cases, including multi-agent systems, long-term memory with extraction strategies, and specialized memory retrieval based on conversation context.

我们演示的方法可以扩展到更复杂的用例，包括多代理系统、带有提取策略的长期记忆以及基于对话上下文的专门记忆检索。

## Clean up

## 清理

Let's delete the memory to clean up the resources used in this notebook.

让我们删除内存以清理本笔记本中使用的资源。

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)